# Real-ESRGAN 4x Batch Upscaler (Google Colab GPU)

Upscale kumpulan gambar secara massal dengan GPU T4.  
- **Model**: RealESRGAN_x4plus (Anti-blur, pertajam detail & tekstur).
- **Format Output**: JPG Kualitas Tinggi (Quality 96, Subsampling 4:4:4, ukuran file padat dan tidak drop).
- **Tiling**: 512x512 (Aman dari CUDA Out-of-Memory untuk gambar besar).

---  
**Pastikan Runtime GPU Aktif**:  
Klik menu `Runtime` -> `Change runtime type` -> pilih `T4 GPU` -> `Save`.

In [ ]:
# 1. Periksa GPU
!nvidia-smi

In [ ]:
# 2. Instal dependensi & download model
print("Memasang dependensi...")
!pip install -q basicsr facexlib gfpgan realesrgan

# Patch torchvision bug di basicsr
!python -c "import site, os; paths = [os.path.join(s, 'basicsr/data/degradations.py') for s in site.getsitepackages() if os.path.exists(os.path.join(s, 'basicsr/data/degradations.py'))]; [open(f, 'w').write(open(f).read().replace('functional_tensor', 'functional')) for f in paths]"

# Buat folder kerja
!mkdir -p weights inputs outputs

# Download model RealESRGAN_x4plus (jika belum ada)
!wget -c -q --show-progress "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" -P weights/
print("Setup selesai!")

In [ ]:
# 3. Siapkan modul runner batch upscaler
%%writefile upscale_batch.py
import argparse
import glob
import os
import cv2
import torch
import numpy as np
from PIL import Image
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

def parse_args():
    parser = argparse.ArgumentParser(description="Batch Image Upscaler Real-ESRGAN")
    parser.add_argument("-i", "--input", type=str, default="inputs", help="Input folder")
    parser.add_argument("-o", "--output", type=str, default="outputs", help="Output folder")
    parser.add_argument("-m", "--model_path", type=str, default="weights/RealESRGAN_x4plus.pth", help="Model path")
    parser.add_argument("-s", "--outscale", type=float, default=4.0, help="Scale factor")
    parser.add_argument("-t", "--tile", type=int, default=512, help="Tile size")
    parser.add_argument("--tile_pad", type=int, default=10, help="Tile padding")
    parser.add_argument("-q", "--quality", type=int, default=96, help="JPEG quality")
    parser.add_argument("--fp16", action="store_true", default=True, help="Half precision")
    return parser.parse_args()

def get_upsampler(model_path, scale=4.0, tile=512, tile_pad=10, fp16=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    return RealESRGANer(
        scale=scale,
        model_path=model_path,
        model=model,
        tile=tile,
        tile_pad=tile_pad,
        pre_pad=0,
        half=fp16 and (device.type == "cuda"),
        device=device,
    )

def save_high_quality_jpg(img_np, out_path, quality=96):
    img_rgb = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    pil_img.save(out_path, format="JPEG", quality=quality, subsampling=0, optimize=True)

def main():
    args = parse_args()
    os.makedirs(args.output, exist_ok=True)
    exts = ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.bmp")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(args.input, ext)))
        files.extend(glob.glob(os.path.join(args.input, ext.upper())))
    files = sorted(list(set(files)))

    if not files:
        print(f"Tidak ada file gambar di '{args.input}'.")
        return

    print(f"Memproses {len(files)} gambar...")
    upsampler = get_upsampler(args.model_path, args.outscale, args.tile, args.tile_pad, args.fp16)

    for i, path in enumerate(files, 1):
        name = os.path.splitext(os.path.basename(path))[0]
        out_path = os.path.join(args.output, f"{name}_upscaled.jpg")
        print(f"[{i}/{len(files)}] {os.path.basename(path)} -> ", end="", flush=True)
        try:
            img = cv2.imread(path, cv2.IMREAD_COLOR)
            if img is None:
                print("GAGAL (Gagal baca gambar)")
                continue
            out, _ = upsampler.enhance(img, outscale=args.outscale)
            save_high_quality_jpg(out, out_path, args.quality)
            print(f"SELESAI ({out.shape[1]}x{out.shape[0]} px)")
        except Exception as e:
            print(f"GAGAL: {e}")

if __name__ == "__main__":
    main()


In [ ]:
# 4. Upload gambar bahan ke folder inputs
# Anda bisa drag-and-drop langsung ke folder 'inputs' di panel sebelah kiri,
# atau gunakan tombol upload di bawah ini:
import os
from google.colab import files

print("Pilih gambar bahan untuk di-upload:")
uploaded = files.upload()
for fn in uploaded.keys():
    os.rename(fn, os.path.join('inputs', fn))
print(f"Berhasil mengunggah {len(uploaded)} gambar ke folder inputs/.")

In [ ]:
# 5. Jalankan Batch Upscaling (4x, Tiling 512, High-Quality JPG)
!python upscale_batch.py --input inputs --output outputs --tile 512 --quality 96

In [ ]:
# 6. Kompres folder outputs & Download hasil
import shutil
from google.colab import files

print("Mengompres hasil ke zip...")
shutil.make_archive("hasil_upscale", "zip", "outputs")
print("Mengunduh hasil_upscale.zip ke komputer Anda...")
files.download("hasil_upscale.zip")